# Baselines

Establishes the two linear baselines reported in the paper: **Linear Regression**
(raw features) and **Polynomial Regression, degree 2**. This is the only place in
the repository where these two numbers are computed -- without this notebook they
are not reproducible.

Uses the **identical** train/test split (seed=42, 80/20) and 5-fold KFold
(seed=42, shuffle=True) as `01_random_forest.ipynb`, `02_neural_network.ipynb`
and `03_model_comparison.ipynb`.

## Split structure
```
Full dataset  (216 samples)
  └─ Test set   (44 samples, 20%)  ← touched once, at the very end
  └─ Train set  (172 samples, 80%) ← used for training + KFold CV
```

In [1]:
import random
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error

# ── Determinism ────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Data loading and shared split

In [2]:
df = pd.read_csv('../data/raw/hepg2.csv')
print(f'Dataset: {df.shape[0]} samples x {df.shape[1]} columns')

X_raw = df[['% DMSO', 'TREHALOSE']].values
y     = df['VIABILIDADE'].values

# ── Shared 80/20 split (identical to the other notebooks) ──
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=SEED
)

# ── Shared KFold (identical to the other notebooks) ──
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

print(f'Train: {X_train_raw.shape[0]} | Test: {X_test_raw.shape[0]}')

Dataset: 216 samples x 6 columns
Train: 172 | Test: 44


## Polynomial feature engineering

Used only by the polynomial baseline below. **Fitted on training data only**
to prevent leakage.

In [3]:
# Polynomial features (degree=2, no bias term because LinearRegression adds it)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_raw)   # fit on train only
X_test_poly  = poly.transform(X_test_raw)

feature_names_poly = poly.get_feature_names_out(['DMSO', 'Trehalose'])
print('Polynomial features:', list(feature_names_poly))

Polynomial features: ['DMSO', 'Trehalose', 'DMSO^2', 'DMSO Trehalose', 'Trehalose^2']


## Baseline 1 — Linear Regression (raw features)

In [4]:
lr = LinearRegression()
lr.fit(X_train_raw, y_train)

y_pred_lr = lr.predict(X_test_raw)
r2_lr_test  = r2_score(y_test, y_pred_lr)
rmse_lr_test = np.sqrt(mean_squared_error(y_test, y_pred_lr))

cv_lr = cross_val_score(lr, X_train_raw, y_train, cv=cv, scoring='r2')
r2_lr_cv = cv_lr.mean()

print(f'Linear Regression (raw): R²(test)={r2_lr_test:.4f} | RMSE(test)={rmse_lr_test:.4f} | R²(CV)={r2_lr_cv:.4f}')

Linear Regression (raw): R²(test)=0.4775 | RMSE(test)=25.5001 | R²(CV)=0.3399


## Baseline 2 — Polynomial Regression degree 2

In [5]:
# Polynomial regression = LinearRegression on PolynomialFeatures
pr = LinearRegression()
pr.fit(X_train_poly, y_train)

y_pred_pr = pr.predict(X_test_poly)
r2_pr_test  = r2_score(y_test, y_pred_pr)
rmse_pr_test = np.sqrt(mean_squared_error(y_test, y_pred_pr))

# CV must use poly-transformed features consistently — build a pipeline
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr',   LinearRegression())
])
cv_pr = cross_val_score(poly_pipeline, X_train_raw, y_train, cv=cv, scoring='r2')
r2_pr_cv = cv_pr.mean()

print(f'Polynomial Regression (deg=2): R²(test)={r2_pr_test:.4f} | RMSE(test)={rmse_pr_test:.4f} | R²(CV)={r2_pr_cv:.4f}')

Polynomial Regression (deg=2): R²(test)=0.6096 | RMSE(test)=22.0430 | R²(CV)=0.3787
